# Location Extraction: Modern LLM Inference

This notebook evaluates modern decoder models (LLMs) for extracting location hierarchies from conflict event descriptions using zero-shot inference (no fine-tuning).

## Models Evaluated

1. **llama-3.1-8b-instruct** - Open-source benchmark (Meta)
2. **mistral-7b-instruct** - Smaller, efficient model (Mistral AI)
3. **mixtral-8x7b-instruct** - Mixture-of-Experts model (Mistral AI)
4. **flan-t5-xl** - Large instruction-following T5 model (Google)
5. **gpt-4o-mini** - Cheap, strong reference (OpenAI API)
6. **gemini-2.5-flash** - Free, fast baseline (Google API)

## Notebook Structure

1. **Setup & Configuration** - Environment setup, API key configuration
2. **Data Management** - Load location data and create temporal splits
3. **Model Configuration** - Setup helper functions and parsing logic
4. **Model Inference** - Run inference on each model and compute metrics
5. **Results Comparison** - Merge results and compare all models

## Notes

- Local models (Llama, Mistral, Mixtral, Flan-T5) run on GPU if available
- API models (GPT-4o-mini, Gemini) require API keys in Colab secrets
- Location extraction uses structured output format: `state: X, district: Y, village: Z, other_locations: W`
- Metrics include exact and fuzzy matching with per-level breakdown

## 1. Setup and Configuration

### 1.1 Environment Setup

#### 1.1.1 Colab Setup

In [ ]:
# 1) Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# 2) Clone or update the repo
BRANCH = "main"  # Change if working on a different branch

# Clone fresh each session 
!rm -rf /content/code-satp
!git clone -b $BRANCH --depth 1 https://github.com/eteitelbaum/code-satp.git /content/code-satp

# 3) Install dependencies from both count-models and location-models
%pip install -qU pip setuptools wheel
%pip install -r /content/code-satp/models/count-models/requirements.txt
%pip install rapidfuzz  # For fuzzy location matching

# 4) Make result directories and add to path
import pathlib
import sys
pathlib.Path("/content/drive/MyDrive/colab/satp-results").mkdir(parents=True, exist_ok=True)
pathlib.Path("/content/drive/MyDrive/colab/satp-results/location-extraction-llms").mkdir(parents=True, exist_ok=True)
pathlib.Path("figures").mkdir(parents=True, exist_ok=True)
sys.path.append("/content/code-satp/models/count-models")
sys.path.append("/content/code-satp/models/location-models")

# 5) GPU check
import torch
print("="*80)
print("✅ SETUP COMPLETE")
print("="*80)
print(f"📂 Cloned repo: /content/code-satp")
if torch.cuda.is_available():
    print(f"🖥️  GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  WARNING: Running on CPU! Inference will be very slow.")
    print("   Go to Runtime > Change runtime type > Hardware accelerator > GPU")
print("="*80)

# 6) Set canonical task name for this notebook
TASK_NAME = "location-extraction-llms"


#### 1.1.2 Local Setup

In [ ]:
# # For local development, uncomment these lines:
# import sys
# import os

# # Add utils to path
# sys.path.append('./utils')
# sys.path.append('../count-models')

# # Local data and results paths
# DATA_PATH = "../../data/"
# RESULTS_PATH = "./results/"

# # Create local results directory
# os.makedirs(RESULTS_PATH, exist_ok=True)
# os.makedirs(os.path.join(RESULTS_PATH, "location-extraction-llms"), exist_ok=True)
# os.makedirs("./figures", exist_ok=True)

# # Verify GPU availability
# import torch
# if torch.cuda.is_available():
#     print(f"✅ GPU Available: {torch.cuda.get_device_name(0)}")
# else:
#     print("⚠️ GPU not available, using CPU")

# # Set task name
# TASK_NAME = "location-extraction-llms"

# print("✅ Local setup complete!")


### 1.2 Import Libraries

In [ ]:
# Core libraries
import os
import gc
import re
import json
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

# Deep learning
import torch

# Setup paths
import sys
sys.path.append("/content/code-satp/models/count-models")
sys.path.insert(0, "/content/code-satp/models/location-models")

# Purge any preloaded 'utils' entries to avoid conflicts (BEFORE importing)
for _k in list(sys.modules.keys()):
    if _k == "utils" or _k.startswith("utils."):
        del sys.modules[_k]

# Count-models utils for loading/timing only (NOT for inference - they apply death-count prompts!)
import utils as cm_utils

# Location-models utilities (includes location-specific inference runners)
from utils.metrics_utils import parse_structured_location, fuzzy_match, print_metrics
from utils.file_io import get_task_results_dir, save_dataframe_csv
from utils.data_utils import build_structured_location
from utils.llm_location_utils import (
    # Prompt creation
    make_location_prompt,
    LOCATION_EXTRACTION_INSTRUCTION,
    # Parsing and metrics
    parse_location_from_llm,
    dict_to_structured_string,
    compute_location_metrics_from_strings,
    print_location_metrics,
    run_and_save_llm_location_results,
    # Location-specific inference runners (don't apply death-count prompts)
    run_location_causal_batch,
    run_location_t5_batch,
    run_location_openai_batch,
    run_location_gemini_batch,
    run_location_gemini_json_batch,  # <- added
)
print("✅ Loaded utilities:")
print("   - cm_utils: Model loading (load_causal, load_t5) and timing (time_inference_call)")
print("   - utils: Location-specific inference runners and metrics")

# Configuration
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    

### 1.3 Hugging Face Authentication (Llama/Mixtral/Flan-T5)

In [ ]:
# Hugging Face token: Colab secrets first, else local env
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('huggingface_token')
    if HF_TOKEN:
        os.environ['HUGGINGFACE_TOKEN'] = HF_TOKEN
        print("✅ Hugging Face token loaded from Colab secrets")
    else:
        print("⚠️  Hugging Face token not found in Colab secrets")
except ImportError:
    HF_TOKEN = os.environ.get('HUGGINGFACE_TOKEN')
    if HF_TOKEN:
        print("✅ Hugging Face token loaded from environment")
    else:
        print("⚠️  Hugging Face token not found in environment")

### 1.4 Open AI and Gemini API Keys

In [ ]:
# Fetch API keys from Colab secrets or environment variables
try:
    # Colab environment – load from secrets panel (🔑 icon)
    from google.colab import userdata

    OPENAI_API_KEY = userdata.get('openai_api_key')
    GEMINI_API_KEY = userdata.get('gemini_api_key')

    print("✅ OpenAI API key loaded from Colab secrets")
    print("✅ Gemini API key loaded from Colab secrets")

except ImportError:
    # Local/offline execution – load from environment variables
    OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY')
    GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY')

    print("✅ OpenAI API key loaded from environment")
    print("✅ Gemini API key loaded from environment")

## 2 Data Management

### 2.1 Load Location Data

Load location data with human annotations for state, district, village, and other locations.

In [ ]:
# Try loading from cloned repository first
try:
    data = pd.read_csv('/content/code-satp/data/location_info_augmented.csv', dtype=str)
    print("✅ Data loaded successfully from cloned repository")
    print(f"Rows: {len(data)}, Columns: {len(data.columns)}")
except Exception as e:
    print(f"❌ Error loading CSV from cloned repo: {e}")
    print("🔧 Fallback: Trying GitHub URL...")
    
    # Fallback to GitHub if cloned repo fails or if working locally
    url = "https://raw.githubusercontent.com/eteitelbaum/code-satp/refs/heads/main/data/location_info_augmented.csv"
    try:
        data = pd.read_csv(url, dtype=str)
        print("✅ Data loaded successfully from GitHub")
        print(f"Rows: {len(data)}, Columns: {len(data.columns)}")
    except Exception as e:
        print(f"❌ Error loading CSV from GitHub: {e}")

### 2.2 Create Temporal Splits

Create train/validation/test splits using temporal ordering (80/10/10). Training ends before Telangana formation (June 2, 2014).

In [ ]:
# Ensure human_annotated_location exists
if 'human_annotated_location' not in data.columns:
    print("Creating human_annotated_location column...")
    data['human_annotated_location'] = data.apply(build_structured_location, axis=1)

# TEMPORAL SPLIT: Sort by date for chronological train/val/test split
data['date'] = pd.to_datetime(data['date'], errors='coerce')
data = data.sort_values('date').reset_index(drop=True)

# Calculate split indices for 80/10/10 split
train_end_idx = int(len(data) * 0.8)
val_end_idx = int(len(data) * 0.9)

# Create temporal splits
train_data = data.iloc[:train_end_idx].copy()
val_data = data.iloc[train_end_idx:val_end_idx].copy()
test_data = data.iloc[val_end_idx:].copy()

print(f"Temporal split summary:")
print(f"  Training:   {len(train_data)} incidents ({train_data['date'].min()} to {train_data['date'].max()})")
print(f"  Validation: {len(val_data)} incidents ({val_data['date'].min()} to {val_data['date'].max()})")
print(f"  Test:       {len(test_data)} incidents ({test_data['date'].min()} to {test_data['date'].max()})")
print(f"\nNote: Training ends before Telangana formation (June 2, 2014)")
print(f"      Test set contains only post-2014 incidents with current state names")

### 2.3 Select and Evaluate Split

Toggle between validation and test splits. Use validation for development, test for final evaluation.

In [ ]:
# Toggle which split to use downstream
EVAL_SPLIT = "test"  # Options: "test" for final inference, "val" for development

if EVAL_SPLIT not in {"test", "val"}:
    raise ValueError("EVAL_SPLIT must be either 'test' or 'val'")

df = test_data.copy() if EVAL_SPLIT == "test" else val_data.copy()
split_label = "test" if EVAL_SPLIT == "test" else "validation"
print(f"\n✅ Using {split_label} split for downstream analysis: {len(df):,} incidents")

# Ensure an ID column exists
if 'incident_number' not in df.columns:
    df = df.reset_index(drop=True).reset_index()
    df['incident_number'] = f'{EVAL_SPLIT}_' + df['index'].astype(str)
    df = df.drop('index', axis=1)

ID_COL = 'incident_number'
print(f"✅ Using '{ID_COL}' as ID column")

# Verify required columns exist
required_cols = ['incident_summary', 'human_annotated_location']
missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

# Show some examples
print("\n" + "="*80)
print(f"Example incidents from {split_label} split:")
print("="*80)
for i, row in df.head(3).iterrows():
    summary = row['incident_summary'][:150] + "..." if len(row['incident_summary']) > 150 else row['incident_summary']
    print(f"\nID: {row[ID_COL]}")
    print(f"Location: {row['human_annotated_location']}")
    print(f"Text: {summary}")
    print("-" * 80)

## 3. Model Configuration and Setup

### 3.1 Test Location Extraction Prompt

The prompt creation is handled by `make_location_prompt()` in `utils/llm_location_utils.py`.

The prompt asks models to extract structured location information in the format:
`state: <name>, district: <name>, village: <name>, other_locations: <name>`

In [ ]:
# Test the location extraction prompt (imported from utils.llm_location_utils)
print("="*80)
print("LOCATION EXTRACTION PROMPT TEMPLATE")
print("="*80)
print(f"\nInstruction (from LOCATION_EXTRACTION_INSTRUCTION):")
print(f"  {LOCATION_EXTRACTION_INSTRUCTION}")

print(f"\n\nFull prompt format (from make_location_prompt()):")
print("-"*80)

# Show example with first incident
example_text = df.iloc[0]['incident_summary'][:150] + "..." if len(df.iloc[0]['incident_summary']) > 150 else df.iloc[0]['incident_summary']
example_prompt = make_location_prompt(example_text)
print(example_prompt)
print("-"*80)

print(f"\n✅ Prompt function loaded from utils.llm_location_utils")
print(f"   This prompt will be used for all {len(df):,} incidents in the dataset")

### 3.2 Test Location Parser

In [ ]:
# Test the location parser (functions imported from utils.llm_location_utils)
test_cases = [
    "state: Chhattisgarh, district: Sukma, village: Dornapal",
    "```state: Andhra Pradesh, district: Visakhapatnam```",
    "The location is: state: Telangana, district: Adilabad, village: Asifabad, other_locations: Kagaznagar"
]

print("Testing location parser (from utils.llm_location_utils):")
print("-" * 80)
for test in test_cases:
    parsed = parse_location_from_llm(test)
    reconstructed = dict_to_structured_string(parsed)
    print(f"Input:  {test[:70]}...")
    print(f"Output: {reconstructed}")
    print()

### 3.3 Test Metrics Computation

In [ ]:
# Test the metrics computation (functions imported from utils.llm_location_utils)
test_preds = [
    "state: Chhattisgarh, district: Sukma, village: Dornapal",
    "state: Andhra Pradesh, district: Visakhapatnam"
]
test_labels = [
    "state: Chhattisgarh, district: Sukma, village: Dornapal",
    "state: Andhra Pradesh, district: Vizianagaram"
]

print("Testing metrics computation (from utils.llm_location_utils):")
print("=" * 80)
test_metrics = compute_location_metrics_from_strings(test_preds, test_labels)
print_location_metrics(test_metrics, "Test")

### 3.4 Implement run_and_save Helper

In [ ]:
# Configure paths and directories
results_dir = get_task_results_dir(TASK_NAME, create=True)
OUTPUT_DIR = results_dir

print(f"📁 Output directory: {OUTPUT_DIR}")
print(f"📋 ID column: {ID_COL}")
print(f"📊 Dataset size: {len(df):,} incidents")

# Create a shorter alias for the runner function (imported from utils.llm_location_utils)
# This allows existing code using run_and_save() to work without changes
run_and_save = lambda model_name, outputs, df_input, id_col, timing=None: \
    run_and_save_llm_location_results(model_name, outputs, df_input, id_col, OUTPUT_DIR, timing)

print("\n✅ Helper functions configured (using utils.llm_location_utils.run_and_save_llm_location_results)")

### 3.5 Performance Configuration

In [ ]:
# ============================================================================
# Performance Configuration for A100 GPU
# ============================================================================

# Disable 4-bit quantization for A100 (fp16 is faster on 40GB RAM)
# For smaller GPUs, keep USE_4BIT = True
cm_utils.USE_4BIT = False

# Batch processing configuration (speeds up inference dramatically)
BATCH_SIZE = 16          # Process 16-32 prompts at once (adjust based on GPU memory)
MAX_INPUT_TOKENS = 512   # Truncate long inputs for speed
MAX_NEW_TOKENS = 64      # Locations are short, don't need many tokens

print("⚡ Performance Configuration:")
print(f"   USE_4BIT: {cm_utils.USE_4BIT} (fp16 on A100 is faster than 4-bit)")
print(f"   BATCH_SIZE: {BATCH_SIZE} (process multiple prompts in parallel)")
print(f"   MAX_INPUT_TOKENS: {MAX_INPUT_TOKENS} (truncate long inputs)")
print(f"   MAX_NEW_TOKENS: {MAX_NEW_TOKENS} (locations are typically short)")
print("\n💡 Expected speedup: ~10-20x faster than sequential processing")
print("   Llama-3 8B: hours → minutes on A100")

## 4. Model Inference

Run inference on 6 LLM models for location extraction.

### 4.1 Llama-3.1-8B-Instruct

In [ ]:
MODEL = "meta-llama/Meta-Llama-3.1-8B-Instruct"
NAME = "llama3_8b"

if not cm_utils.llm_already_done(NAME, OUTPUT_DIR):
    print(f"\n{'='*80}")
    print(f"Running {NAME}...")
    print(f"{'='*80}")
    
    # Load model
    print("Loading model...")
    tok, mdl = cm_utils.load_causal(MODEL)
    
    # Create prompts
    texts = [make_location_prompt(summary) for summary in df['incident_summary'].tolist()]
    
    # Run inference WITH TIMING (using location-specific runner - no death-count prompt!)
    print("Running inference...")
    outs, timing = cm_utils.time_inference_call(
        run_location_causal_batch, 
        tok, mdl, texts, 
        max_new_tokens=MAX_NEW_TOKENS,
        max_input_tokens=MAX_INPUT_TOKENS,
        batch_size=BATCH_SIZE
    )
    
    # Save results and compute metrics (with timing)
    results_df, metrics = run_and_save(NAME, outs, df, ID_COL, timing=timing)
    
    # Clean up GPU memory
    del tok, mdl
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
else:
    print(f"✅ {NAME}: Already completed (loading results...)")
    results_df = pd.read_csv(OUTPUT_DIR / f"{NAME}.csv")
    print(f"   Loaded {len(results_df)} results")


### 4.2 Mistral-7B-Instruct-v0.3

In [ ]:
MODEL = "mistralai/Mistral-7B-Instruct-v0.3"
NAME = "mistral_7b"

if not cm_utils.llm_already_done(NAME, OUTPUT_DIR):
    print(f"\n{'='*80}")
    print(f"Running {NAME}...")
    print(f"{'='*80}")
    
    # Load model
    print("Loading model...")
    tok, mdl = cm_utils.load_causal(MODEL)
    
    # Create prompts
    texts = [make_location_prompt(summary) for summary in df['incident_summary'].tolist()]
    
    # Run inference WITH TIMING (using location-specific runner - no death-count prompt!)
    print("Running inference...")
    outs, timing = cm_utils.time_inference_call(
        run_location_causal_batch, 
        tok, mdl, texts, 
        max_new_tokens=MAX_NEW_TOKENS,
        max_input_tokens=MAX_INPUT_TOKENS,
        batch_size=BATCH_SIZE
    )
    
    # Save results and compute metrics (with timing)
    results_df, metrics = run_and_save(NAME, outs, df, ID_COL, timing=timing)
    
    # Clean up GPU memory
    del tok, mdl
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
else:
    print(f"✅ {NAME}: Already completed (loading results...)")
    results_df = pd.read_csv(OUTPUT_DIR / f"{NAME}.csv")
    print(f"   Loaded {len(results_df)} results")


### 4.3 Mixtral-8×7B-Instruct-v0.1

In [ ]:
# Note: Mixtral is a large model - may require significant GPU memory
MODEL = "mistralai/Mixtral-8x7B-Instruct-v0.1"
NAME = "mixtral_8x7b"

if not cm_utils.llm_already_done(NAME, OUTPUT_DIR):
    print(f"\n{'='*80}")
    print(f"Running {NAME}...")
    print(f"{'='*80}")
    print("⚠️  Warning: This is a large model. May require significant GPU memory.")
    
    # Load model
    print("Loading model...")
    tok, mdl = cm_utils.load_causal(MODEL)
    
    # Create prompts
    texts = [make_location_prompt(summary) for summary in df['incident_summary'].tolist()]
    
    # Run inference WITH TIMING (using location-specific runner - no death-count prompt!)
    print("Running inference...")
    outs, timing = cm_utils.time_inference_call(
        run_location_causal_batch, 
        tok, mdl, texts, 
        max_new_tokens=MAX_NEW_TOKENS,
        max_input_tokens=MAX_INPUT_TOKENS,
        batch_size=BATCH_SIZE
    )
    
    # Save results and compute metrics (with timing)
    results_df, metrics = run_and_save(NAME, outs, df, ID_COL, timing=timing)
    
    # Clean up GPU memory
    del tok, mdl
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
else:
    print(f"✅ {NAME}: Already completed (loading results...)")
    results_df = pd.read_csv(OUTPUT_DIR / f"{NAME}.csv")
    print(f"   Loaded {len(results_df)} results")


### 4.4 Flan-T5-XL

In [ ]:
MODEL = "google/flan-t5-xl"
NAME = "flan_t5_xl"

if not cm_utils.llm_already_done(NAME, OUTPUT_DIR):
    print(f"\n{'='*80}")
    print(f"Running {NAME}...")
    print(f"{'='*80}")
    
    # Load model
    print("Loading model...")
    tok, mdl = cm_utils.load_t5(MODEL)
    
    # Create prompts
    texts = [make_location_prompt(summary) for summary in df['incident_summary'].tolist()]
    
    # Run inference WITH TIMING (using location-specific runner - no death-count prompt!)
    print("Running inference...")
    outs, timing = cm_utils.time_inference_call(
        run_location_t5_batch, 
        tok, mdl, texts, 
        max_new_tokens=MAX_NEW_TOKENS,
        max_input_tokens=MAX_INPUT_TOKENS,
        batch_size=BATCH_SIZE
    )
    
    # Save results and compute metrics (with timing)
    results_df, metrics = run_and_save(NAME, outs, df, ID_COL, timing=timing)
    
    # Clean up GPU memory
    del tok, mdl
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
else:
    print(f"✅ {NAME}: Already completed (loading results...)")
    results_df = pd.read_csv(OUTPUT_DIR / f"{NAME}.csv")
    print(f"   Loaded {len(results_df)} results")


### 4.5 GPT-4o-mini

In [ ]:
NAME = "gpt4o_mini"

if not cm_utils.llm_already_done(NAME, OUTPUT_DIR):
    print(f"\n{'='*80}")
    print(f"Running {NAME}...")
    print(f"{'='*80}")
    
    if not OPENAI_API_KEY:
        print("⚠️  ERROR: OpenAI API key not found!")
        print("   Please add 'openai_api_key' to Colab secrets or set OPENAI_API_KEY environment variable")
    else:
        # Create prompts
        texts = [make_location_prompt(summary) for summary in df['incident_summary'].tolist()]
        
        # Run inference WITH TIMING (using location-specific runner - no death-count prompt!)
        print("Running inference via OpenAI API...")
        outs, timing = cm_utils.time_inference_call(
            run_location_openai_batch,
            texts,
            api_key=OPENAI_API_KEY,
            model_name="gpt-4o-mini",
            max_tokens=128,  # Locations are short, 128 is sufficient
            rate_limit_delay=0.1
        )
        
        # Save results and compute metrics (with timing)
        results_df, metrics = run_and_save(NAME, outs, df, ID_COL, timing=timing)
else:
    print(f"✅ {NAME}: Already completed (loading results...)")
    results_df = pd.read_csv(OUTPUT_DIR / f"{NAME}.csv")
    print(f"   Loaded {len(results_df)} results")


### 4.6 Gemini-2.5-Flash

In [ ]:
# NAME = "gemini_flash"
# GETTING ERRORS DUE TO SAFETY FILTERS

# if not cm_utils.llm_already_done(NAME, OUTPUT_DIR):
#     print(f"\n{'='*80}")
#     print(f"Running {NAME}...")
#     print(f"{'='*80}")
    
#     if not GEMINI_API_KEY:
#         print("⚠️  ERROR: Gemini API key not found!")
#         print("   Please add 'gemini_api_key' to Colab secrets or set GEMINI_API_KEY environment variable")
#     else:
#         # Create prompts
#         texts = [make_location_prompt(summary) for summary in df['incident_summary'].tolist()]
        
#         # Replace the old runner call with this JSON-mode call
#         outs, timing = cm_utils.time_inference_call(
#             run_location_gemini_json_batch,
#             texts,
#             api_key=GEMINI_API_KEY,
#             model_name="models/gemini-2.5-flash",
#             max_output_tokens=512,
#             rate_limit_delay=0.1,
#             max_concurrency=8,
#             max_chars=350,
#         )
#         # Save + evaluate (unchanged)
#         results_df, metrics = run_and_save(NAME, outs, df, ID_COL, timing=timing)
# else:
#     print(f"✅ {NAME}: Already completed (loading results...)")
#     results_df = pd.read_csv(OUTPUT_DIR / f"{NAME}.csv")
#     print(f"   Loaded {len(results_df)} results")

## 5. Merge Completed Runs for Comparison

In [ ]:
# Start with base dataframe (IDs and true labels)
merged = df[[ID_COL, 'incident_summary', 'human_annotated_location']].copy()
merged = merged.rename(columns={'human_annotated_location': 'true_location'})

# Ensure ID column is string type for consistent merging
merged[ID_COL] = merged[ID_COL].astype(str)

# Model names to merge
model_names = ['llama3_8b', 'mistral_7b', 'mixtral_8x7b', 'flan_t5_xl', 'gpt4o_mini', 'gemini_flash']

# Bring in each model file that exists
print(f"\n{'='*80}")
print("Merging results from all models...")
print(f"{'='*80}")

merged_count = 0
for model_name in model_names:
    model_file = OUTPUT_DIR / f"{model_name}.csv"
    if model_file.exists():
        try:
            tmp = pd.read_csv(model_file)
            # Convert ID column to string to match merged DataFrame
            tmp[ID_COL] = tmp[ID_COL].astype(str)
            # Keep prediction column
            pred_col = f'{model_name}_prediction'
            
            if pred_col in tmp.columns:
                tmp = tmp[[ID_COL, pred_col]].rename(columns={pred_col: f'{model_name}_pred'})
                merged = merged.merge(tmp, on=ID_COL, how='left')
                merged_count += 1
                print(f"✅ Merged {model_name}")
            else:
                print(f"⚠️  {model_name}: Could not find prediction column")
        except Exception as e:
            print(f"❌ {model_name}: Error merging - {e}")
    else:
        print(f"⏭️  {model_name}: Results file not found, skipping")

print(f"\n✅ Merged {merged_count} models")
print(f"   Total rows: {len(merged)}")
print(f"   Columns: {merged.columns.tolist()}")

# Save merged results
merged_path = OUTPUT_DIR / "merged_results.csv"
merged.to_csv(merged_path, index=False)
print(f"\n✅ Saved merged results to: {merged_path}")

# Compute comparison metrics
print(f"\n{'='*80}")
print("Model Comparison Summary")
print(f"{'='*80}")

comparison_data = []
for model_name in model_names:
    pred_col = f'{model_name}_pred'
    if pred_col in merged.columns:
        # Filter out NaN predictions
        mask = merged[pred_col].notna()
        if mask.sum() > 0:
            preds = merged.loc[mask, pred_col].tolist()
            labels = merged.loc[mask, 'true_location'].tolist()
            
            # Compute metrics
            metrics = compute_location_metrics_from_strings(preds, labels)
            overall = metrics.get('overall', {})
            
            comparison_data.append({
                'Model': model_name,
                'Exact Match': overall.get('exact_match', 0),
                'Exact Core': overall.get('exact_core_match', 0),
                'Fuzzy Match': overall.get('fuzzy_match', 0),
                'Fuzzy Core': overall.get('fuzzy_core_match', 0),
                'Micro Exact F1': overall.get('micro_exact_f1', 0),
                'Micro Fuzzy F1': overall.get('micro_fuzzy_f1', 0),
                'N': len(preds)
            })

if comparison_data:
    comparison_df = pd.DataFrame(comparison_data)
    print("\n" + comparison_df.to_string(index=False))
    
    # Save comparison
    comparison_path = OUTPUT_DIR / "model_comparison.csv"
    comparison_df.to_csv(comparison_path, index=False)
    print(f"\n✅ Saved comparison metrics to: {comparison_path}")
else:
    print("⚠️  No model results found for comparison")

# Display merged results head
print(f"\n{'='*80}")
print("Merged Results Preview (first 10 rows)")
print(f"{'='*80}")
print(merged.head(10).to_string())
